# 01 — Build the Dislocation Universe

Scan for U.S. equities with **500%+ 3-day moves** or **1000%+ moves**.

Pipeline:
1. Seed list scan (known extreme movers)
2. SEC EDGAR full universe scan
3. FTD signal scan (squeeze candidates)
4. Enrich with fundamentals
5. Classify into taxonomy buckets

**Prerequisites:** Start the API first:
```bash
cd apps/api && uvicorn app.main:app --reload --port 8000
```

In [1]:
import requests
import pandas as pd

API = "http://localhost:8000/quant/research"

def api(method, path, **kwargs):
    resp = getattr(requests, method)(f"{API}{path}", **kwargs)
    resp.raise_for_status()
    return resp.json()

/Users/erichurchey/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## 1. View the seed list

In [3]:
seeds = api("get", "/seed-list")
print(f"Seed list: {seeds['total']} known extreme movers")
print(f"Catalyst types: {seeds['catalysts']}")
pd.DataFrame(seeds["events"])[["symbol", "name", "catalyst", "notes"]]

Seed list: 35 known extreme movers
Catalyst types: ['biotech_catalyst', 'earnings_surprise', 'merger_acquisition', 'penny_stock_pump', 'restructuring', 'sector_momentum', 'short_squeeze', 'spac_despac']


,symbol,name,catalyst,notes
0,GME,GameStop,short_squeeze,"WSB-driven short squeeze, 140%+ SI, Melvin Cap..."
1,AMC,AMC Entertainment,short_squeeze,"Meme stock squeeze, high retail participation"
2,SPRT,Support.com,short_squeeze,"Merger with Greenidge, high SI, low float squeeze"
3,IRNT,IronNet Cybersecurity,short_squeeze,"De-SPAC gamma squeeze, tiny float post-merger"
4,BBIG,Vinco Ventures,short_squeeze,"Tyde spinoff, short squeeze setup"
5,ATER,Aterian,short_squeeze,"Low float, high SI squeeze"
6,BGFV,Big 5 Sporting,short_squeeze,Special dividend + short squeeze
7,ISPC,iSpecimen,short_squeeze,"Low float squeeze, COVID testing demand"
8,DWAC,Digital World Acquisition (Trump Media),spac_despac,"Trump Media SPAC announcement, political meme ..."
9,BKKT,Bakkt Holdings,spac_despac,"De-SPAC pop, Mastercard partnership rumors"


## 2. Scan seed list for dislocations
This scans ~35 known tickers via yfinance. Takes a few minutes.

In [9]:
result = api("post", "/build-seeds?start_date=2019-01-01&min_return_3d_pct=500")
print(f"Scanned: {result['tickers_scanned']} tickers")
print(f"Events persisted: {result['events_persisted']}")
if result["errors"]:
    print(f"Errors ({len(result['errors'])}): {result['errors'][:5]}")

Scanned: 35 tickers
Events persisted: 0


## 3. Check dataset stats

In [10]:
stats = api("get", "/stats")
print(f"Total events:    {stats['total_events']}")
print(f"Label A (500%+): {stats['label_a_count']}")
print(f"Label B (1000%+):{stats['label_b_count']}")
print(f"Continuation rate: {stats['continuation_rate']}")
print(f"\nBucket distribution:")
for bucket, count in sorted(stats["bucket_distribution"].items(), key=lambda x: -x[1]):
    print(f"  {bucket}: {count}")

Total events:    15
Label A (500%+): 15
Label B (1000%+):15
Continuation rate: 0.0667

Bucket distribution:
  short_squeeze: 10
  penny_stock_pump: 2
  spac_despac: 2
  biotech_catalyst: 1


## 4. Browse events

In [11]:
events = api("get", "/events?limit=50")
df = pd.DataFrame(events)
cols = ["symbol", "company_name", "return_3d_pct", "return_1d_pct", "label_a", "label_b",
        "day_after_continuation", "volume_ratio", "taxonomy_bucket", "event_start_date"]
df[cols].sort_values("return_3d_pct", ascending=False)

,symbol,company_name,return_3d_pct,return_1d_pct,label_a,label_b,day_after_continuation,volume_ratio,taxonomy_bucket,event_start_date
0,BBIG,None,3860.00,1880.00,1,1,0,3.57,short_squeeze,2025-11-05T00:00:00
1,BBIG,None,2900.00,48.15,1,1,0,2.35,short_squeeze,2024-12-31T00:00:00
2,BBIG,None,2900.00,2900.00,1,1,0,0.16,short_squeeze,2025-01-15T00:00:00
3,BBIG,None,2150.00,2900.00,1,1,0,1.23,short_squeeze,2025-05-06T00:00:00
4,BBIG,None,1566.67,0.00,1,1,0,0.00,short_squeeze,2024-07-03T00:00:00
5,TOP,None,1512.67,441.05,1,1,0,8.99,penny_stock_pump,2023-04-25T00:00:00
6,HKD,None,867.53,85.38,1,1,1,0.41,penny_stock_pump,2022-07-27T00:00:00
7,DJT,None,841.06,107.03,1,1,0,1433.30,spac_despac,2021-10-19T00:00:00
8,OCGN,None,784.35,222.98,1,1,0,65.01,biotech_catalyst,2020-12-18T00:00:00
9,PHUN,None,756.86,471.24,1,1,0,647.09,spac_despac,2021-10-19T00:00:00


## 5. Enrich events with fundamentals
Pulls float, short interest, sector, market cap from yfinance.

In [17]:
enrich_result = api("post", "/enrich", json={})
print(f"Events enriched: {enrich_result['events_enriched']}")
if enrich_result["errors"]:
    print(f"Errors: {enrich_result['errors'][:5]}")

Events enriched: 0


## 6. Classify into taxonomy buckets

In [18]:
classify_result = api("post", "/classify", json={})
print(f"Events classified: {classify_result['events_classified']}")
print(f"Bucket counts: {classify_result['bucket_counts']}")

Events classified: 0
Bucket counts: {}


## 7. Export full dataset

In [19]:
# Load full enriched + classified dataset
events = api("get", "/events?limit=1000")
df = pd.DataFrame(events)
print(f"Dataset: {len(df)} events")
print(f"\nColumns: {list(df.columns)}")
df.describe()

Dataset: 92 events

Columns: ['id', 'symbol', 'company_name', 'security_type', 'event_start_date', 'event_end_date', 'peak_date', 'price_start', 'price_peak', 'price_end_3d', 'return_1d_pct', 'return_3d_pct', 'return_peak_pct', 'label_a', 'label_b', 'day_after_continuation', 'volume_event_day', 'volume_avg_20d_pre', 'volume_ratio', 'shares_outstanding', 'float_shares', 'market_cap_pre', 'short_interest_shares', 'short_pct_float', 'days_to_cover', 'sector', 'industry', 'exchange', 'ipo_date', 'days_since_ipo', 'catalyst_summary', 'news_count_event_day', 'options_available', 'iv_pre_event', 'call_oi_pre', 'put_oi_pre', 'taxonomy_bucket', 'taxonomy_confidence', 'taxonomy_notes', 'data_source', 'created_at']


,id,price_start,price_peak,price_end_3d,return_1d_pct,return_3d_pct,return_peak_pct,label_a,label_b,day_after_continuation,...,volume_ratio,shares_outstanding,float_shares,market_cap_pre,short_interest_shares,short_pct_float,days_to_cover,days_since_ipo,options_available,taxonomy_confidence
count,92.00000,92.000000,92.000000,92.000000,92.000000,92.000000,92.000000,92.0,92.000000,91.000000,...,92.000000,9.200000e+01,9.200000e+01,9.200000e+01,9.200000e+01,92.000000,92.000000,3.000000,92.000000,9.200000e+01
mean,46.50000,14.011139,68.186930,50.747114,240.440435,379.767283,540.169130,1.0,0.228261,0.120879,...,79.665652,1.436040e+08,8.827056e+07,3.070014e+09,1.358194e+07,0.132198,2.918804,7320.000000,0.260870,9.000000e-01
std,26.70206,39.345736,189.739979,139.596862,513.021012,625.014592,870.016758,0.0,0.422011,0.327793,...,250.600441,1.815942e+08,1.293716e+08,1.641967e+10,2.653239e+07,0.085345,3.411539,206.574442,0.441515,4.465226e-16
min,1.00000,0.000100,0.000200,0.000200,0.000000,-81.940000,0.000000,1.0,0.000000,0.000000,...,0.000000,4.064050e+06,3.173944e+06,1.300000e+03,3.687300e+04,0.000900,0.080000,7096.000000,0.000000,9.000000e-01
25%,23.75000,0.000675,0.004000,0.003300,35.025000,111.577500,144.280000,1.0,0.000000,0.000000,...,0.625000,1.300000e+07,1.166532e+07,2.340000e+04,1.940990e+06,0.021700,1.230000,7228.500000,0.000000,9.000000e-01
50%,46.50000,0.055500,0.205000,0.170000,100.000000,182.475000,285.115000,1.0,0.000000,0.000000,...,3.800000,1.659408e+07,1.166532e+07,1.748421e+06,2.108418e+06,0.162200,1.230000,7361.000000,0.000000,9.000000e-01
75%,69.25000,10.002500,37.990000,29.237500,206.600000,333.172500,468.220000,1.0,0.000000,0.000000,...,20.777500,2.895228e+08,1.057062e+08,7.446616e+08,3.901087e+06,0.162200,3.305000,7432.000000,1.000000,9.000000e-01
max,92.00000,265.200000,1202.000000,765.000000,2900.000000,3860.000000,5900.000000,1.0,1.000000,1.000000,...,1433.300000,5.827972e+08,5.246068e+08,1.545578e+11,1.180756e+08,0.314600,12.790000,7503.000000,1.000000,9.000000e-01


In [20]:
# Save to CSV
df.to_csv("../data/dislocations.csv", index=False)
print("Saved to data/dislocations.csv")

Saved to data/dislocations.csv


## 8. (Optional) Expand to EDGAR universe
This scans thousands of tickers — use `max_tickers` to limit while testing.

In [ ]:
# Start small — scan 100 EDGAR tickers
# edgar_result = api("post", "/build-edgar?max_tickers=100&start_date=2019-01-01")
# print(edgar_result)

In [ ]:
# Check FTD signals
# ftd = api("get", "/ftd-signals?year=2024&half=1")
# pd.DataFrame(ftd["symbols"]).head(20)